# Test Parquet Data Files
This notebook tests the parquet files created by fetch_market_data.py

In [9]:
import pandas as pd
import os
from pathlib import Path
import json

## Check Available Files

In [ ]:
# List all parquet files in data directory
data_dir = Path('data')

print("Data Directory Structure:")
print("=" * 50)

for ticker in ['iwm', 'spy', 'qqq', 'spx']:
    ticker_dir = data_dir / ticker
    if ticker_dir.exists():
        print(f"\n{ticker.upper()}:")
        
        # Daily files
        daily_files = list(ticker_dir.glob('*.parquet'))
        for f in sorted(daily_files):
            size_kb = f.stat().st_size / 1024
            print(f"  {f.name} ({size_kb:.1f} KB)")
        
        # Summary files
        summary_files = list(ticker_dir.glob('*.json'))
        for f in sorted(summary_files):
            size_kb = f.stat().st_size / 1024
            print(f"  {f.name} ({size_kb:.1f} KB)")
        
        # Minute files
        minute_dir = ticker_dir / 'minute'
        if minute_dir.exists():
            minute_files = list(minute_dir.glob('*.parquet'))
            print(f"  Minute data: {len(minute_files)} files")
            if minute_files:
                # Show first and last
                sorted_files = sorted(minute_files)
                first = sorted_files[0]
                last = sorted_files[-1]
                print(f"    First: {first.name}")
                print(f"    Last:  {last.name}")
    else:
        print(f"\n{ticker.upper()}: Directory not found")

## Test Daily Data

In [ ]:
# Test reading a daily data file
ticker = 'spy'  # Change this to test different tickers: iwm, spy, qqq, spx
ticker_dir = data_dir / ticker
daily_file = ticker_dir / f"{ticker}_2025.parquet"

if daily_file.exists():
    df_daily = pd.read_parquet(daily_file)
    
    print(f"=== {ticker.upper()} Daily Data ===")
    print(f"Shape: {df_daily.shape}")
    print(f"Date Range: {df_daily.index.min()} to {df_daily.index.max()}")
    print(f"\nColumns ({len(df_daily.columns)}):")
    print(list(df_daily.columns))
    
    print("\nFirst 3 rows:")
    display(df_daily.head(3))
    
    print("\nLast 3 rows:")
    display(df_daily.tail(3))
    
    # Check for expected columns
    expected_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    missing = [col for col in expected_cols if col not in df_daily.columns]
    if missing:
        print(f"\n⚠️ WARNING: Missing expected columns: {missing}")
    else:
        print(f"\n✅ All expected OHLCV columns present")
    
    # Check for indicators
    indicator_cols = ['rsi_14', 'ma_20', 'ema_9', 'atr_14', 'rvol']
    found_indicators = [col for col in indicator_cols if col in df_daily.columns]
    if found_indicators:
        print(f"✅ Found indicators: {found_indicators}")
    
else:
    print(f"File not found: {daily_file}")

## Test Minute Data

In [ ]:
# Test reading a minute data file
ticker = 'spy'  # Change this to test different tickers
date = '20250829'  # Change this to test different dates

ticker_dir = data_dir / ticker
minute_dir = ticker_dir / 'minute'
minute_file = minute_dir / f"{ticker}_minute_{date}.parquet"

if minute_file.exists():
    df_minute = pd.read_parquet(minute_file)
    
    print(f"=== {ticker.upper()} Minute Data for {date} ===")
    print(f"Shape: {df_minute.shape}")
    print(f"Time Range: {df_minute.index.min()} to {df_minute.index.max()}")
    
    # Check columns
    print(f"\nColumns: {list(df_minute.columns)}")
    print(f"Column types: {type(df_minute.columns).__name__}")
    
    # Check for MultiIndex columns (which would be a problem)
    if isinstance(df_minute.columns, pd.MultiIndex):
        print("\n⚠️ WARNING: MultiIndex columns detected!")
        print(f"Level 0: {list(df_minute.columns.get_level_values(0).unique())}")
        print(f"Level 1: {list(df_minute.columns.get_level_values(1).unique())}")
    else:
        print("✅ Single-level columns (correct)")
    
    print("\nFirst 5 rows:")
    display(df_minute.head())
    
    print("\nLast 5 rows:")
    display(df_minute.tail())
    
    # Statistics
    print("\nDaily Statistics from Minute Data:")
    if 'Open' in df_minute.columns:
        print(f"  Open: ${df_minute['Open'].iloc[0]:.2f}")
    if 'High' in df_minute.columns:
        print(f"  High: ${df_minute['High'].max():.2f}")
    if 'Low' in df_minute.columns:
        print(f"  Low: ${df_minute['Low'].min():.2f}")
    if 'Close' in df_minute.columns:
        print(f"  Close: ${df_minute['Close'].iloc[-1]:.2f}")
    if 'Volume' in df_minute.columns:
        print(f"  Total Volume: {df_minute['Volume'].sum():,.0f}")
        print(f"  Avg Volume/min: {df_minute['Volume'].mean():,.0f}")
else:
    print(f"File not found: {minute_file}")
    print("\nAvailable minute files:")
    if minute_dir.exists():
        for f in sorted(minute_dir.glob(f"{ticker}_minute_*.parquet"))[:5]:
            print(f"  {f.name}")

## Test Summary JSON

In [ ]:
# Test reading summary JSON
ticker = 'spy'  # Change this to test different tickers
ticker_dir = data_dir / ticker
summary_file = ticker_dir / f"{ticker}_summary.json"

if summary_file.exists():
    with open(summary_file, 'r') as f:
        summary = json.load(f)
    
    print(f"=== {ticker.upper()} Summary ===")
    for key, value in summary.items():
        if isinstance(value, float):
            if 'return' in key.lower() or 'pct' in key.lower():
                print(f"{key}: {value:.2f}%")
            elif 'volume' in key.lower():
                print(f"{key}: {value:,.0f}")
            else:
                print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")
else:
    print(f"Summary file not found: {summary_file}")

## Validate Data Integrity

In [ ]:
# Comprehensive data validation
def validate_ticker_data(ticker):
    issues = []
    
    # Check ticker directory
    ticker_dir = data_dir / ticker
    if not ticker_dir.exists():
        issues.append(f"Missing ticker directory: {ticker_dir}")
        return issues
    
    # Check daily data
    daily_file = ticker_dir / f"{ticker}_2025.parquet"
    if not daily_file.exists():
        issues.append(f"Missing daily data file: {daily_file.name}")
    else:
        df = pd.read_parquet(daily_file)
        
        # Check required columns
        required = ['Open', 'High', 'Low', 'Close', 'Volume']
        missing = [col for col in required if col not in df.columns]
        if missing:
            issues.append(f"Daily data missing columns: {missing}")
        
        # Check for nulls in OHLCV
        for col in ['Open', 'High', 'Low', 'Close']:
            if col in df.columns and df[col].isna().any():
                issues.append(f"Daily data has NaN values in {col}")
        
        # Check data consistency
        if 'High' in df.columns and 'Low' in df.columns:
            invalid = df[df['High'] < df['Low']]
            if not invalid.empty:
                issues.append(f"Daily data has {len(invalid)} rows where High < Low")
    
    # Check minute data
    minute_dir = ticker_dir / 'minute'
    if not minute_dir.exists():
        issues.append(f"Missing minute directory: {minute_dir}")
    else:
        minute_files = list(minute_dir.glob(f"{ticker}_minute_*.parquet"))
        if not minute_files:
            issues.append(f"No minute data files found for {ticker}")
        else:
            # Check a sample minute file
            sample_file = minute_files[0]
            df_min = pd.read_parquet(sample_file)
            
            if isinstance(df_min.columns, pd.MultiIndex):
                issues.append(f"Minute data has MultiIndex columns (should be flattened)")
            
            missing = [col for col in required if col not in df_min.columns]
            if missing:
                issues.append(f"Minute data missing columns: {missing}")
    
    # Check summary
    summary_file = ticker_dir / f"{ticker}_summary.json"
    if not summary_file.exists():
        issues.append(f"Missing summary file: {summary_file.name}")
    
    return issues

# Validate all tickers
print("=== Data Validation Report ===")
for ticker in ['iwm', 'spy', 'qqq', 'spx']:
    issues = validate_ticker_data(ticker)
    if issues:
        print(f"\n❌ {ticker.upper()}: {len(issues)} issues found")
        for issue in issues:
            print(f"   - {issue}")
    else:
        print(f"\n✅ {ticker.upper()}: All checks passed")

## Quick Data Visualization

In [ ]:
import matplotlib.pyplot as plt

# Plot recent price data for all tickers
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Recent Market Data (Last 5 Days)', fontsize=14)

tickers = ['iwm', 'spy', 'qqq', 'spx']
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    ax = axes[i]
    ticker_dir = data_dir / ticker
    daily_file = ticker_dir / f"{ticker}_2025.parquet"
    
    if daily_file.exists():
        df = pd.read_parquet(daily_file)
        
        # Plot last 5 days
        recent = df.tail(5)
        
        if 'Close' in recent.columns:
            ax.plot(recent.index, recent['Close'], marker='o', label='Close')
            ax.fill_between(recent.index, recent['Low'], recent['High'], alpha=0.3, label='High-Low Range')
            
            ax.set_title(f"{ticker.upper()}")
            ax.set_ylabel('Price ($)')
            ax.grid(True, alpha=0.3)
            ax.legend()
            
            # Format x-axis
            ax.tick_params(axis='x', rotation=45)
        else:
            ax.text(0.5, 0.5, f'No price data for {ticker.upper()}', 
                   ha='center', va='center', transform=ax.transAxes)
    else:
        ax.text(0.5, 0.5, f'No data file for {ticker.upper()}', 
               ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()

## Compare Daily Aggregated vs Minute Data

In [ ]:
# Compare daily OHLCV calculated from minute data vs stored daily data
ticker = 'spy'
test_date = '2025-08-29'

# Load daily data
ticker_dir = data_dir / ticker
daily_file = ticker_dir / f"{ticker}_2025.parquet"

if daily_file.exists():
    df_daily = pd.read_parquet(daily_file)
    
    # Get the row for test_date
    if test_date in df_daily.index.strftime('%Y-%m-%d'):
        daily_row = df_daily[df_daily.index.strftime('%Y-%m-%d') == test_date].iloc[0]
        
        print(f"=== {ticker.upper()} Daily Data for {test_date} ===")
        print(f"Open:  ${daily_row['Open']:.2f}")
        print(f"High:  ${daily_row['High']:.2f}")
        print(f"Low:   ${daily_row['Low']:.2f}")
        print(f"Close: ${daily_row['Close']:.2f}")
        print(f"Volume: {daily_row['Volume']:,.0f}")

# Load minute data for same date
minute_dir = ticker_dir / 'minute'
minute_file = minute_dir / f"{ticker}_minute_{test_date.replace('-', '')}.parquet"

if minute_file.exists():
    df_minute = pd.read_parquet(minute_file)
    
    print(f"\n=== Calculated from Minute Data ===")
    if all(col in df_minute.columns for col in ['Open', 'High', 'Low', 'Close', 'Volume']):
        print(f"Open:  ${df_minute['Open'].iloc[0]:.2f} (first minute)")
        print(f"High:  ${df_minute['High'].max():.2f} (max of all minutes)")
        print(f"Low:   ${df_minute['Low'].min():.2f} (min of all minutes)")
        print(f"Close: ${df_minute['Close'].iloc[-1]:.2f} (last minute)")
        print(f"Volume: {df_minute['Volume'].sum():,.0f} (sum of all minutes)")
        
        print(f"\nMinute bars in file: {len(df_minute)}")
        print(f"Expected (6.5 hours * 60): 390")
    else:
        print("Missing required columns in minute data")
else:
    print(f"\nMinute file not found: {minute_file.name}")